# Exploratory analyses of existing algorithms on the KickCap dataset
*Set Kernel as the kickcap environment you installed thanks to the `environment.yml` file at the project root.*

From this notebook, you can preprocess the data.

Then, this notebook runs the command lines needed to train the P2P-Insole algorithm from Watanabe et al. (2025), to predict with the trained model and to visualize the prediction compared to ground-truth data.

We put the same model with a big conceptual difference in the inference procedure: Watanabe et al. (2025) use a sequence-to-frame inference (prediction of 1 frame based on the n past frames of the inputed window), we want to see if a sequence-to-sequence (all frames of the window are predicted at once) produces different results *(spoiler: no, it's still shit)*.

The final part of the notebook is dedicated to a custom code replicating the SolePoser pipeline (Wu et al., 2024), based on a two-stream cross-attention double-cycle loss Transformer architecture. 

*(Why did we do this: because the simple Transformer from Watanabe et al. (2025) was not able to learn a dynamical mapping between feet pressure and IMU information and whole-body kinematics information. Results: the comparison and, before that, all the tests on the P2P-Insole architecture seem to indicate that the algorithm should be aimed at capturing fine and complex links to be able to learn this mapping in the KickCap dataset. It essentially means that KickCap shows very complex kinematics chains and dynamics, that can only be correctly infered through subtile algorithms, not necessarily with more parameters. Indeed, SoleFormer produces way better results while having less than half the number of parameters as P2P-Insole.)*

The data provided represent <1% of the full KickCap dataset. Apart from enabling the repository to be submitted to Moodle, this allows fast (but bad) training to allow the user to experience the full pipeline in a relatively short time and with lower computing power.

**Important notes**
1. The tags for files calling are the ones for the minimal data provided. Change if needed.
2. If you run the whole notebook, be prepared to wait a certain time *(~3,5 h, ~70 min/model mode with an AMD Ryzen 7 8840HS w/ Radeon 780M Graphics processor)* if you use the minimalist data and if your computer does not have a dedicated graphics card. 
3. I recommend that you use an Nvidia (cuda-compatible) graphics card.

In [ ]:
from sources.util import (
    ablation_row_label,
    add_ablation_flags,
    ensure_runtime_data_ready,
    format_ablation_tag,
    initialize_notebook_runtime,
    join_nonempty,
    normalize_abl_id,
    print_csv_table,
    run_cmd_streaming,
)

Robust path finder and real-time output updates in notebook

In [ ]:
ROOT, PYTHON_CMD = initialize_notebook_runtime()

Repository root: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel
Using Python: c:\Users\joels\anaconda3\python.exe
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


## Data pre-processing

In [ ]:
ensure_runtime_data_ready(ROOT, PYTHON_CMD)

Preprocessing outputs already detected in runtime folders.
Skipping re-run. Delete/refresh runtime files if you want a fresh preprocessing pass.


From then, run either the `Original P2P-Insole` part, the `Simple sequence-to-sequence` part or the `SoleFormer` part, that are based on three different models to compare their specificities: sequence-to-frame versus sequence-to-sequence inference versus two-stream double-loss cross-attention Transformers. In each part, cells must be run in order, because posterior cells rely on produced files from anterior ones.

## Original P2P-Insole Transformer model

In this mode, additional features can be added (add the wanted instruction in the command lines): 
- time feature: `--use_time_feature <true/false>`
- gradient features: `--use_gradient_data <true/false>`

P2P-Insole training

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "train", "--model_mode", "original"], cwd=ROOT)
print("Model file in results/weight/best_skeleton_model_original.pth")

P2P-Insole prediction

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "predict", "--model_mode", "original", "--checkpoint_file", "results/weight/best_skeleton_model_original.pth"], cwd=ROOT)
print("Prediction file in results/output/Predicted_skeleton_<tag>_original.csv")

P2P-Insole visualization

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "visual", "--model_mode", "original", "--real_file", "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv", "--pred_file", "results/output/Predicted_skeleton_001CcSs_3_shadow_test_original.csv"], cwd=ROOT)
print("HTML visualization file in results/animation/Animation_<tag>_original.html")

## Simple sequence-to-sequence Transformer model
*The original Transformer model Watanabe et al. use is a sequence-to-frame model, that computes only the last frame of the inputed window from all previous frames of the same window. This one directly infer frames of one sequence all at once. The difference lies in the importance we accord to past pressure and feet IMU frames for the prediction of the current frame state.*

In this mode, additional features can be added (add the wanted instruction in the command lines): 
- time feature: `--use_time_feature <true/false>`
- gradient features: `--use_gradient_data <true/false>`

Seq2seq training

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "train", "--model_mode", "simple_seq2seq"], cwd=ROOT)
print("Model file in results/weight/best_skeleton_model_simple_seq2seq.pth")

Seq2seq prediction

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "predict", "--model_mode", "simple_seq2seq", "--checkpoint_file", "results/weight/best_skeleton_model_simple_seq2seq.pth"], cwd=ROOT)
print("Prediction file in results/output/Predicted_skeleton_<tag>_simple_seq2seq.csv")

Seq2seq visualization

In [ ]:
run_cmd_streaming([PYTHON_CMD, "-u", "main.py", "visual", "--model_mode", "simple_seq2seq", "--real_file", "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv", "--pred_file", "results/output/Predicted_skeleton_001CcSs_3_shadow_test_simple_seq2seq.csv"], cwd=ROOT)
print("HTML visualization file in results/animation/Animation_<tag>_simple_seq2seq.html")

## SoleFormer two-stream double-loss cycle cross-attention Transformer
For this algorithm that produces much better results (not an unmoving skeleton), we provide options for ablation. Meaning, the default is the original SoleFormer pipeline, whereas ablation of certain parts can be overrided in the commands to compare the importance of each algorithm part. If you add an overriding instruction in training, you need to add it in the prediction commandline as well (and the visualization, to get the right checkpoint).

### *Ablation commandlines*

In [ ]:
ABL_COMMANDS_CSV = ROOT / "data" / "ablation_commands.csv"
print_csv_table(ABL_COMMANDS_CSV)

abl_id | Category   | Ablation                                    | CommandLine                                                                                                                                                                                                             | Notes                                                                                                                       
1      | Cycle      | Full SoleFormer default                     | python main.py train --model_mode soleformer                                                                                                                                                                            | Default code path: use_cycle_loss=true - enable both cycle branche - pretrain auxiliaries - freeze them during main training
2      | Cycle      | No cycle loss                               | python main.py train --model_mode soleformer --use_cycle_loss false                                           

SoleFormer training

In [ ]:
ABL_ID = None  # Set this to the ablation row id you want to run.
abl_tag = format_ablation_tag(ABL_ID)
abl_arg = normalize_abl_id(ABL_ID)

train_cmd = [PYTHON_CMD, "-u", "main.py", "train", "--model_mode", "soleformer"]
if abl_arg:
    train_cmd.extend(["--abl_id", abl_arg])

train_cmd, train_added_tokens, train_ablation_row = add_ablation_flags(
    "train",
    train_cmd,
    abl_id=ABL_ID,
    csv_path=ABL_COMMANDS_CSV,
)

if train_ablation_row:
    print("Selected ablation:", ablation_row_label(train_ablation_row))
if train_added_tokens:
    print("Added ablation flags to train:", " ".join(train_added_tokens))

run_cmd_streaming(train_cmd, cwd=ROOT)
print(f"Model file in results/weight/{join_nonempty('best_skeleton_model', abl_tag, 'soleformer')}.pth")

SoleFormer prediction

In [ ]:
abl_tag = format_ablation_tag(ABL_ID)
abl_arg = normalize_abl_id(ABL_ID)
checkpoint_name = join_nonempty("best_skeleton_model", abl_tag, "soleformer")
pred_stem = join_nonempty(abl_tag, "001CcSs_3_shadow_test", "soleformer")

predict_cmd = [
    PYTHON_CMD,
    "-u",
    "main.py",
    "predict",
    "--model_mode",
    "soleformer",
    "--checkpoint_file",
    f"results/weight/{checkpoint_name}.pth",
]
if abl_arg:
    predict_cmd.extend(["--abl_id", abl_arg])

predict_cmd, predict_added_tokens, predict_ablation_row = add_ablation_flags(
    "predict",
    predict_cmd,
    abl_id=ABL_ID,
    csv_path=ABL_COMMANDS_CSV,
)

if predict_ablation_row:
    print("Selected ablation:", ablation_row_label(predict_ablation_row))
if predict_added_tokens:
    print("Added ablation flags to predict:", " ".join(predict_added_tokens))

run_cmd_streaming(predict_cmd, cwd=ROOT)
print(f"Prediction file in results/output/Predicted_skeleton_{pred_stem}.csv")

SoleFormer visualization

In [ ]:
abl_tag = format_ablation_tag(ABL_ID)
abl_arg = normalize_abl_id(ABL_ID)
pred_stem = join_nonempty(abl_tag, "001CcSs_3_shadow_test", "soleformer")
pred_file = f"results/output/Predicted_skeleton_{pred_stem}.csv"

visual_cmd = [
    PYTHON_CMD,
    "-u",
    "main.py",
    "visual",
    "--model_mode",
    "soleformer",
    "--real_file",
    "data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv",
    "--pred_file",
    pred_file,
]
if abl_arg:
    visual_cmd.extend(["--abl_id", abl_arg])

visual_cmd, visual_added_tokens, visual_ablation_row = add_ablation_flags(
    "visual",
    visual_cmd,
    abl_id=ABL_ID,
    csv_path=ABL_COMMANDS_CSV,
)

if visual_ablation_row:
    print("Selected ablation:", ablation_row_label(visual_ablation_row))
if visual_added_tokens:
    print("Added ablation flags to visual:", " ".join(visual_added_tokens))

run_cmd_streaming(visual_cmd, cwd=ROOT)
print(f"HTML visualization file in results/animation/Animation_{pred_stem}.html")

Running: c:\Users\joels\anaconda3\python.exe -u main.py visual --model_mode soleformer --real_file data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv --pred_file results/output/Predicted_skeleton_001CcSs_3_shadow_test_soleformer.csv
<load config>
Loading configuration from: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\notebooks\config\transformer_encoder\visual.yaml
Applied CLI overrides:
- visual.model_mode: soleformer -> soleformer
- visual.pred_file: None -> results/output/Predicted_skeleton_001CcSs_3_shadow_test_soleformer.csv
- visual.real_file: None -> data/test_data/skeleton/Awinda_001CcSs_3_shadow_test.csv
Using real skeleton file: data\test_data\skeleton\Awinda_001CcSs_3_shadow_test.csv
Using predicted skeleton file: results\output\Predicted_skeleton_001CcSs_3_shadow_test_soleformer.csv
Real data shape: (501, 71)
Pred data shape: (470, 136)
num frames: 470
Play frame duration (ms): 50
Default FPS: 20
Visualization successful. Saved to: c:\Users\j